# LLM Assisted scRNA seq and Spatial Transcriptomics Analysis of Mouse Brain

This notebook integrates a mouse brain scRNA seq reference with a Visium mouse brain spatial dataset.

## Goals
1. Perform data driven QC for the Zeisel mouse brain scRNA seq dataset.
2. Use Gemini to suggest conservative QC thresholds.
3. Cluster cells and identify marker genes.
4. Run a first pass rule based annotation.
5. Use Gemini to review annotation quality and catch biologically implausible labels.
6. Refine the marker dictionary and rerun annotation.
7. Score corrected cell type signatures in spatial transcriptomics spots.
8. Use Gemini to summarize tissue composition and limitations.

## Notes
- The LLM is used as an interpretation and validation layer, not as the primary clustering or statistics engine.
- Final cluster labels should always be checked against marker genes and biological context.


## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
from google import genai
from dotenv import load_dotenv

In [ ]:
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY", "")
print("Gemini API key loaded:", bool(gemini_api_key))

def get_llm_client():
    if not gemini_api_key:
        return None
    return genai.Client(api_key=gemini_api_key)

def ask_llm(prompt_text, model="gemini-2.5-flash"):
    client = get_llm_client()
    if client is None:
        return "No Gemini API key found. Add GEMINI_API_KEY to your .env file."

    system_prompt = (
        "You are a careful computational biologist. "
        "Use only the information provided. "
        "Do not invent genes, pathways, or cell types. "
        "Explain likely biological identity, marker evidence, uncertainty, "
        "and likely biological role in concise scientific language."
    )

    try:
        full_prompt = f"{system_prompt}\n\n{prompt_text}"
        response = client.models.generate_content(
            model=model,
            contents=full_prompt
        )
        return response.text
    except Exception as e:
        return f"LLM call failed: {e}" 

## 2. Load data

In [ ]:
BRAIN_MARKERS = {
    "Neuron_inhibitory": ["Gad1", "Gad2", "Slc32a1"],
    "Neuron_excitatory": ["Slc17a7", "Slc17a6", "Camk2a", "Tbr1", "Neurod6", "Gria1", "Gria2"],
    "Oligodendrocyte": ["Mbp", "Mog", "Plp1", "Mal", "Ugt8a", "Aspa"],
    "OPC": ["Pdgfra", "Cspg4", "Olig1", "Olig2"],
    "Astrocyte": ["Slc1a3", "Slc1a2", "Aqp4", "Gfap", "Aldh1l1", "Atp1a2", "Gja1"],
    "Microglia": ["Cx3cr1", "P2ry12", "Aif1", "C1qa", "C1qb", "Tyrobp"],
    "Endothelial": ["Pecam1", "Kdr", "Flt1", "Emcn", "Klf2"],
}

In [ ]:
adata_sc = sc.read_h5ad("zeisel_mouse_brain.h5ad")
adata_sc.var_names_make_unique()

adata_sp = sq.datasets.visium(sample_id="V1_Adult_Mouse_Brain")
adata_sp.var_names_make_unique()

print("scRNA dataset:")
print(adata_sc)
print("\nSpatial dataset:")
print(adata_sp)

## 3. scRNA seq QC

In [ ]:
adata_sc_qc = adata_sc.copy()

sc.pp.filter_genes(adata_sc_qc, min_cells=3)
adata_sc_qc.var["mt"] = adata_sc_qc.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata_sc_qc, qc_vars=["mt"], inplace=True)

print(adata_sc_qc)
adata_sc_qc.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe()

In [ ]:
qc_quantiles = adata_sc_qc.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].quantile(
    [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)
qc_quantiles

In [ ]:
sc.pl.violin(
    adata_sc_qc,
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True
)

sc.pl.scatter(adata_sc_qc, x="total_counts", y="n_genes_by_counts")
sc.pl.scatter(adata_sc_qc, x="total_counts", y="pct_counts_mt")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
adata_sc_qc.obs["n_genes_by_counts"].hist(bins=50, ax=axes[0])
axes[0].set_title("Genes detected per cell")
adata_sc_qc.obs["total_counts"].hist(bins=50, ax=axes[1])
axes[1].set_title("Total counts per cell")
adata_sc_qc.obs["pct_counts_mt"].hist(bins=50, ax=axes[2])
axes[2].set_title("Mitochondrial percentage per cell")
plt.tight_layout()
plt.show()

### Gemini assisted QC threshold recommendation

In [ ]:
n_cells = adata_sc_qc.n_obs
n_genes = adata_sc_qc.n_vars

qc_summary_text = f"""
Dataset: Zeisel mouse brain scRNA-seq
Number of cells: {n_cells}
Number of genes: {n_genes}

QC summary statistics:
{adata_sc_qc.obs[['total_counts', 'n_genes_by_counts', 'pct_counts_mt']].describe().to_string()}

QC quantiles:
{qc_quantiles.to_string()}

Please suggest conservative QC thresholds for:
1. minimum genes per cell
2. maximum genes per cell
3. maximum mitochondrial percentage

Important context:
- this is a mouse brain scRNA-seq dataset
- I want conservative filtering, not aggressive filtering
- explain the reasoning and uncertainty
- mention whether the dataset may already be somewhat preprocessed
"""

print(qc_summary_text)
qc_llm_response = ask_llm(qc_summary_text)
print(qc_llm_response)

In [ ]:
adata_sc_filt = adata_sc_qc[
    (adata_sc_qc.obs["n_genes_by_counts"] >= 1000) &
    (adata_sc_qc.obs["n_genes_by_counts"] <= 7500) &
    (adata_sc_qc.obs["pct_counts_mt"] <= 20),
    :
].copy()

print(adata_sc_filt)
print("Cells retained:", adata_sc_filt.n_obs)
print("Cells removed:", adata_sc_qc.n_obs - adata_sc_filt.n_obs)

In [ ]:
adata_sc_filt.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe()

In [ ]:
sc.pl.violin(
    adata_sc_filt,
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True
)

## 4. scRNA seq preprocessing, dimensionality reduction, and clustering

In [ ]:
sc.pp.normalize_total(adata_sc_filt, target_sum=1e4)
sc.pp.log1p(adata_sc_filt)

# Save the full log-normalized matrix for downstream annotation before HVG subsetting.
adata_sc_filt.raw = adata_sc_filt.copy()

sc.pp.highly_variable_genes(adata_sc_filt, n_top_genes=2000, flavor="seurat")
adata_sc_filt = adata_sc_filt[:, adata_sc_filt.var["highly_variable"]].copy()

sc.pp.scale(adata_sc_filt, max_value=10)
sc.tl.pca(adata_sc_filt, svd_solver="arpack")

# Elbow plot used to choose the number of PCs.
sc.pl.pca_variance_ratio(adata_sc_filt, log=False, n_pcs=15)

The elbow plot flattened around 10 PCs, so downstream graph construction used `n_pcs=10`.

In [ ]:
sc.pp.neighbors(adata_sc_filt, n_pcs=10)
sc.tl.umap(adata_sc_filt)
sc.tl.leiden(adata_sc_filt, resolution=0.5, key_added="cluster")

print(adata_sc_filt)
sc.pl.pca(adata_sc_filt, color="cluster")
sc.pl.umap(adata_sc_filt, color="cluster")

## 5. Marker genes and first pass annotation

In [ ]:
sc.tl.rank_genes_groups(adata_sc_filt, groupby="cluster", method="wilcoxon")
sc.pl.rank_genes_groups(adata_sc_filt, n_genes=20, sharey=False)
sc.pl.rank_genes_groups_heatmap(adata_sc_filt, groupby="cluster", n_genes=5)

In [ ]:
result = adata_sc_filt.uns["rank_genes_groups"]
groups = result["names"].dtype.names

marker_rows = []
for group in groups:
    for i in range(10):
        marker_rows.append({
            "cluster": group,
            "gene": result["names"][group][i],
            "score": float(result["scores"][group][i]),
            "logfoldchange": float(result["logfoldchanges"][group][i]) if "logfoldchanges" in result else np.nan,
            "pval_adj": float(result["pvals_adj"][group][i]) if "pvals_adj" in result else np.nan
        })

markers_df = pd.DataFrame(marker_rows)
markers_df.head(20)

In [ ]:
def annotate_clusters(adata_expr, marker_dict, cluster_key="cluster", label_col="predicted_cell_type"):
    cluster_ids = sorted(adata_expr.obs[cluster_key].astype(str).unique())
    annotation_rows = []

    for cluster_id in cluster_ids:
        subset = adata_expr[adata_expr.obs[cluster_key].astype(str) == cluster_id].copy()
        scores = {}

        for celltype, genes in marker_dict.items():
            present = [g for g in genes if g in subset.var_names]
            if len(present) == 0:
                scores[celltype] = np.nan
                continue

            expr = subset[:, present].X
            if hasattr(expr, "toarray"):
                expr = expr.toarray()

            scores[celltype] = float(np.mean(expr))

        valid_scores = {k: v for k, v in scores.items() if pd.notna(v)}
        best_label = max(valid_scores, key=valid_scores.get)

        annotation_rows.append({
            "cluster": cluster_id,
            label_col: best_label,
            **scores
        })

    return pd.DataFrame(annotation_rows)

In [ ]:
# Use the full log-normalized matrix for annotation, not the scaled HVG matrix.
adata_sc_annot = adata_sc_filt.raw.to_adata().copy()
adata_sc_annot.obs["cluster"] = adata_sc_filt.obs["cluster"].values

annotation_df = annotate_clusters(
    adata_sc_annot,
    BRAIN_MARKERS,
    cluster_key="cluster",
    label_col="predicted_cell_type"
)
annotation_df

In [ ]:
cluster_to_label = dict(zip(annotation_df["cluster"], annotation_df["predicted_cell_type"]))
adata_sc_filt.obs["predicted_cell_type"] = adata_sc_filt.obs["cluster"].astype(str).map(cluster_to_label)

sc.pl.umap(adata_sc_filt, color=["cluster", "predicted_cell_type"])

## 6. Gemini review of first pass annotation

This first pass used a limited marker dictionary for broad brain cell classes.

Gemini review was used to check whether the rule based labels were biologically plausible. This step intentionally preserves the first pass labels because it demonstrates a realistic analysis workflow where the initial annotation can be improved after biological review.


In [ ]:
def build_cluster_prompt(markers_df, annotation_df, cluster_id, label_col="predicted_cell_type"):
    cluster_markers = markers_df[markers_df["cluster"].astype(str) == str(cluster_id)]
    ann = annotation_df[annotation_df["cluster"].astype(str) == str(cluster_id)]

    marker_list = ", ".join(cluster_markers["gene"].head(10).tolist())
    pred = ann[label_col].iloc[0] if len(ann) else "Unknown"

    return f"""
Dataset: Zeisel mouse brain scRNA-seq
Cluster: {cluster_id}
Rule-based predicted cell type: {pred}
Top marker genes: {marker_list}

Please do the following:
1. Evaluate whether the predicted cell type is reasonable.
2. Suggest 1 to 2 possible alternative identities if relevant.
3. Explain which markers support the interpretation.
4. Mention uncertainty and limitations.
5. Keep the explanation concise and scientifically cautious.
"""

In [ ]:
test_clusters = annotation_df["cluster"].astype(str).tolist()

for cluster_id in test_clusters[:5]:
    print("=" * 80)
    print(f"CLUSTER {cluster_id}")
    print("=" * 80)
    prompt_text = build_cluster_prompt(markers_df, annotation_df, cluster_id)
    response = ask_llm(prompt_text)
    print(response)
    print("\n")

## 7. Corrected annotation after Gemini review

Gemini identified that some first pass assignments were biologically implausible. For example, a cluster initially labeled as oligodendrocyte showed strong contractile mural cell markers such as `Acta2`, `Myh11`, `Tagln`, `Myl9`, and `Mylk`.

This suggested that the original marker dictionary was incomplete rather than the clustering itself being wrong. The dictionary was therefore expanded and annotation was rerun.


In [ ]:
BRAIN_MARKERS_V2 = {
    "Neuron_inhibitory": ["Gad1", "Gad2", "Slc32a1"],
    "Neuron_excitatory": ["Slc17a7", "Slc17a6", "Camk2a", "Tbr1", "Neurod6", "Gria1", "Gria2"],
    "Oligodendrocyte": ["Mbp", "Mog", "Plp1", "Mal", "Ugt8a", "Aspa", "Ermn"],
    "OPC": ["Pdgfra", "Cspg4", "Olig1", "Olig2"],
    "Astrocyte": ["Slc1a3", "Slc1a2", "Aqp4", "Gfap", "Aldh1l1", "Atp1a2", "Gja1", "Gpr37l1"],
    "Microglia": ["Cx3cr1", "P2ry12", "Aif1", "C1qa", "C1qb", "Tyrobp"],
    "Endothelial": ["Pecam1", "Kdr", "Flt1", "Emcn", "Klf2"],
    "Pericyte_VSMC": ["Acta2", "Myh11", "Tagln", "Myl9", "Mylk", "Tpm1", "Tpm2", "Rgs5"],
    "Ependymal": ["Foxj1", "Dynlrb2", "Pifo", "Tmem212"],
}

In [ ]:
annotation_df_v2 = annotate_clusters(
    adata_sc_annot,
    BRAIN_MARKERS_V2,
    cluster_key="cluster",
    label_col="predicted_cell_type_v2"
)
annotation_df_v2

In [ ]:
annotation_compare = annotation_df.merge(
    annotation_df_v2[["cluster", "predicted_cell_type_v2"]],
    on="cluster",
    how="left"
)
annotation_compare

In [ ]:
cluster_to_label_v2 = dict(zip(annotation_df_v2["cluster"], annotation_df_v2["predicted_cell_type_v2"]))
adata_sc_filt.obs["predicted_cell_type_v2"] = adata_sc_filt.obs["cluster"].astype(str).map(cluster_to_label_v2)

sc.pl.umap(adata_sc_filt, color=["cluster", "predicted_cell_type", "predicted_cell_type_v2"])

In [ ]:
markers_with_labels = markers_df.merge(
    annotation_df_v2[["cluster", "predicted_cell_type_v2"]],
    on="cluster",
    how="left"
)
markers_with_labels.head(30)

In [ ]:
def build_cluster_prompt_v2(markers_df, annotation_df_v2, cluster_id):
    cluster_markers = markers_df[markers_df["cluster"].astype(str) == str(cluster_id)]
    ann = annotation_df_v2[annotation_df_v2["cluster"].astype(str) == str(cluster_id)]

    marker_list = ", ".join(cluster_markers["gene"].head(10).tolist())
    pred = ann["predicted_cell_type_v2"].iloc[0] if len(ann) else "Unknown"

    return f"""
Dataset: Zeisel mouse brain scRNA-seq
Cluster: {cluster_id}
Corrected rule-based predicted cell type: {pred}
Top marker genes: {marker_list}

Please do the following:
1. Evaluate whether the corrected predicted cell type is reasonable.
2. Suggest 1 to 2 possible alternative identities if relevant.
3. Explain which markers support the interpretation.
4. Mention uncertainty and limitations.
5. Keep the explanation concise and scientifically cautious.
"""

In [ ]:
for cluster_id in ["0", "1", "10", "11", "2"]:
    print("=" * 80)
    print(f"CLUSTER {cluster_id}")
    print("=" * 80)
    prompt_text_v2 = build_cluster_prompt_v2(markers_df, annotation_df_v2, cluster_id)
    print(ask_llm(prompt_text_v2))
    print("\n")

## 8. Spatial transcriptomics preprocessing

In [ ]:
adata_sp_proc = adata_sp.copy()

adata_sp_proc.var["mt"] = adata_sp_proc.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata_sp_proc, qc_vars=["mt"], inplace=True)

if "in_tissue" in adata_sp_proc.obs.columns:
    adata_sp_proc = adata_sp_proc[adata_sp_proc.obs["in_tissue"] == 1].copy()

sc.pp.normalize_total(adata_sp_proc, target_sum=1e4)
sc.pp.log1p(adata_sp_proc)

adata_sp_proc.raw = adata_sp_proc.copy()

sc.pp.highly_variable_genes(adata_sp_proc, n_top_genes=2000, flavor="seurat")
adata_sp_proc = adata_sp_proc[:, adata_sp_proc.var["highly_variable"]].copy()

sc.pp.scale(adata_sp_proc, max_value=10)
sc.tl.pca(adata_sp_proc, svd_solver="arpack")
sc.pp.neighbors(adata_sp_proc, n_pcs=10)
sc.tl.umap(adata_sp_proc)
sc.tl.leiden(adata_sp_proc, resolution=0.4, key_added="spatial_cluster")

print(adata_sp_proc)
sc.pl.umap(adata_sp_proc, color="spatial_cluster")
sq.pl.spatial_scatter(adata_sp_proc, color="spatial_cluster")

## 9. Score corrected cell type signatures in spatial transcriptomics

In [ ]:
for celltype, genes in BRAIN_MARKERS_V2.items():
    valid_genes = [g for g in genes if g in adata_sp_proc.var_names]
    if len(valid_genes) > 0:
        sc.tl.score_genes(
            adata_sp_proc,
            gene_list=valid_genes,
            score_name=f"{celltype}_score"
        )

score_cols = [c for c in adata_sp_proc.obs.columns if c.endswith("_score")]
adata_sp_proc.obs["dominant_celltype_v2"] = (
    adata_sp_proc.obs[score_cols].idxmax(axis=1).str.replace("_score", "", regex=False)
)

adata_sp_proc.obs[["dominant_celltype_v2"] + score_cols].head()

In [ ]:
sq.pl.spatial_scatter(adata_sp_proc, color="dominant_celltype_v2")

In [ ]:
spatial_counts_v2 = adata_sp_proc.obs["dominant_celltype_v2"].value_counts().reset_index()
spatial_counts_v2.columns = ["cell_type", "n_spots"]
spatial_counts_v2["percent"] = 100 * spatial_counts_v2["n_spots"] / spatial_counts_v2["n_spots"].sum()
spatial_counts_v2

## 10. Gemini summary of spatial tissue composition

In [ ]:
spatial_prompt_v2 = (
    "The following dominant cell type assignments were observed across spatial transcriptomic spots "
    "in a mouse brain sample after corrected rule-based annotation:\n\n"
    f"{spatial_counts_v2.to_string(index=False)}\n\n"
    "Please summarize what this suggests about tissue composition, major cell populations, "
    "and limitations of spot based spatial transcriptomics."
)

print(spatial_prompt_v2)
spatial_llm_response_v2 = ask_llm(spatial_prompt_v2)
print(spatial_llm_response_v2)

## 11. Save key outputs

In [ ]:
annotation_df_v2.to_csv("zeisel_corrected_annotation.csv", index=False)
annotation_compare.to_csv("zeisel_annotation_comparison.csv", index=False)
spatial_counts_v2.to_csv("spatial_celltype_summary.csv", index=False)

print("Saved:")
print("- zeisel_corrected_annotation.csv")
print("- zeisel_annotation_comparison.csv")
print("- spatial_celltype_summary.csv")